In [1]:
#setting libraries
import os
import re
from datetime import *
import itertools
import numpy as np
import time

Data overview

In [ ]:
#reading data and separating data into test and bench files
tests_files=[a for a in os.listdir('./data.nogit/') if 'test' in a]
bench_files=[a for a in os.listdir('./data.nogit/') if 'bench' in a]
# os.getcwd()
#overview of the test files
bits={} #key:file name value:number of bits


for circ_struct in tests_files:
    total_bits=0
    n_input=0
    n_outpus=0
    with open(f'./data.nogit/{circ_struct}','r') as file:
        while True:
            line=file.readline().strip()
            if not(line):
                break
            else: #there's available data to read
                total_bits=max(total_bits,len(line))
                n_outpus=line.count('-')
                n_input=len(line)-n_outpus
    bits[circ_struct]={"total_bits":total_bits,"INPUT":n_input,"OUTPUT":n_outpus}
    file.close()

for circ_struct,circ_struc in bits.items():
    print(f'file: {circ_struct}, total bits to work: {circ_struc}')
def extract_number_from_string(string:str):
    '''
    :param str string: the string of values to evaluate
    input string to extract the integer value
    '''
    print(string)
    aux=re.findall(r'(\d+) \w+',string)
    print(aux)
    return int(aux[0])
#overview of bench files
n_vales={} #key:file_name value:{key: input/output/gates value:read value from file}
exp_values={} #key: input/output/gates value:expected number
for circ_struct in bench_files:
    n_vales[circ_struct]={}
    exp_values[circ_struct]={}
    aux=0
    print(f'read file: {circ_struct}')
    with open(f'./data.nogit/{circ_struct}','r') as file:
        while True:
            line=file.readline().strip()
            if aux>3:
                break
            if not(line):
                aux+=1           
                continue
            else:
                if line[0]=='#': #it show the information about the file
                    try:
                        exp_values[circ_struct][line.split(' ')[2].upper()]=extract_number_from_string(line.lower())
                    except:
                        continue
                else:
                    try:
                        gate_str="".join(filter(str.isalpha,line.split('=')[1].split('(')[0]))
                    except:    
                        gate_str="".join(filter(str.isalpha,line.split('(')[0]))
                    if gate_str not in n_vales[circ_struct]:
                        n_vales[circ_struct][gate_str]=0
                    n_vales[circ_struct][gate_str]+=1
    file.close()
print('|File|Primary Inputs (PIs)|Primary Outputs (POs)|Logic gates|')
print("|---"*4,'|')
gates=[] #getting the names of the gates used in the bench files
for circ_struct,circ_struc in n_vales.items():
    # print (circ_struct,circ_struc)
    print (circ_struct,end="|")
    aux=0 #count for logic gates
    for hierarchy  in circ_struc.keys():
        if hierarchy not in ['INPUT','OUTPUT']:
            if hierarchy not in gates:
                gates.append(hierarchy)
            aux+=circ_struc[hierarchy]
        else: #printing the inputs and outputs
            print(circ_struc[hierarchy],end="|")
    print(aux,end="|")
    print()

# print(gates)

Code for running tests

In [2]:
#setting time to Japan time (UTC+9)
jpTime=datetime.now(timezone(timedelta(hours=9)))
#when testing started
t0=jpTime.now()
def log(msg:str):
    '''
    :param str msg: the message to appear in the log
    '''
    delta=jpTime.now()-t0 #show the running time
    print(f'{delta}...{msg}')
def logic_gate(gate:str, inputs:list): 

    '''defining the logic gate functioning 

    Params
    ------
        gate(str): 'NAND', 'AND', 'OR', 'NOT', 'NOR', 'BUFF', 'XOR', 'XNOR' name of gates in bench files 
        inputs(list): defines the inputs for the gate [A,B,C...], the number of elements in the function will determine the amount of pins for the logic gate.EXCEPT for NOT gates it will take ONLY the first element 

    Returns
    -------
    output(bool): 
        a single logic value to assign to the corresponding  

    ''' 

    #verifying that the input list is just integers 0/1 
    aux,inputs=inputs,[] 
    try: #typical inputs of 0 or 1
        inputs=[int(x) for x in aux] 
        holder=inputs[0] 
        if gate.upper()=='NAND': 
            [holder:=holder&x for x in inputs] 
            holder= ~holder
        elif gate.upper()=="AND":
            [holder:=holder&x for x in inputs] 
        elif gate.upper()=="OR": 
            holder =0 
            [holder:=holder|x for x in inputs] 
        elif gate.upper()=="NOT":
            holder = ~holder
        elif gate.upper()=="NOR": 
            holder =0 
            [holder:=holder|x for x in inputs] 
            holder=~holder
        elif gate.upper()=="XOR": 
            holder=0 
            [holder:=holder^x for x in inputs] 
        elif gate.upper()=="XNOR": 
            holder=0 
            [holder:=holder^x for x in inputs] 
            holder=~holder
    except: #for forward implication (having X, D, -D)
        if 'X' in aux or 'x' in aux:
            if gate.upper()=='NAND': 
                if 0 in aux or '0' in aux:
                    holder=1
                else:
                    holder='X'
            elif gate.upper()=="AND":
                if 0 in aux or '0' in aux:
                    holder=0
                else:
                    holder='X'
            elif gate.upper()=="OR": 
                if 1 in aux or '1' in aux:
                    holder=1
                else:
                    holder='X'
            elif gate.upper()=="NOR": 
                if 1 in aux or '1' in aux:
                    holder=0
                else:
                    holder='X'
            elif gate.upper()=="NOT" or gate.upper()=="BUFF":
                holder = 'X'
            elif gate.upper()=="XOR" or gate.upper()=="XNOR":
                holder='X'
        ##Pending to integrate the rest of forward implication table for D and -D
        elif 'D' in aux or 'd' in aux: 
            if gate.upper()=='NAND': 
                if 0 in aux or '0' in aux:
                    holder=1
                else:
                    holder='X'
            elif gate.upper()=="AND":
                if 0 in aux or '0' in aux:
                    holder=0
                else:
                    holder='X'
            elif gate.upper()=="OR": 
                if 1 in aux or '1' in aux:
                    holder=1
                else:
                    holder='X'
            elif gate.upper()=="NOR": 
                if 1 in aux or '1' in aux:
                    holder=0
                else:
                    holder='X'
            elif gate.upper()=="NOT":
                holder = 'X'
            elif gate.upper()=="XOR" or gate.upper()=="XNOR":
                holder='X'

    return holder
#circuit code
def circuit_code(signal_line:dict, circ_struc:list, stuck_at:dict={}):
    """This runs the circuit in fault-free mode and stuck-at fault

    Args:
        signal_line (dict): Imports what the signal logic value must be
        circ_struc (list): Which logic gate and the relationship between inputs and outputs
        stuck_at (dict): If provided, these values are flipped to whichever value is in this dict #key=signal index value=logic value to be stuck at
    Returns:
        signal_line (dict): returns the updated signal logic value for any situation (fault-free or stuck-at fault)
    """
    for a in circ_struc:
        output_signal=a.split(' = ')[0]
        gate=a.split(' = ')[1].split('(')[0]
        input_signal_index=a.split(' = ')[1].split('(')[1][:-1]

        input_signals=[]
        for x in input_signal_index.split(', '):
            if x in stuck_at:
                 input_signals.append(stuck_at[x.strip()])
            else:
                input_signals.append(signal_line[x.strip()])
        signal_line[output_signal]=logic_gate(gate,input_signals)
        # if stuck_at and (output_signal in stuck_at.keys()):
        if stuck_at:
            if (output_signal in stuck_at.keys()): #continue with the SA fault
                signal_line[output_signal]=stuck_at[output_signal]
    return signal_line

def bin2int(primary_inputs:list):
    """Transforms the binary functional truth table to unsigned integer for PPSFP application

    Args:
        primary_inputs (list): primary inputs of the circuit

    Returns:
        representative_int(dict): Key:primary input Value:integer representation
    """    
    representative_int={}
    for a in primary_inputs:
        representative_int[a]=[]
    for a in range(2**len(primary_inputs)):
        aux=0
        c=bin(a)[2:].zfill(len(primary_inputs)) #transform to binary 
        while aux<len(primary_inputs):
            representative_int[primary_inputs[aux]].append(c[aux])
            aux+=1
    for a in representative_int.keys():
        representative_int[a]=int(''.join(representative_int[a]),2)
    return representative_int

def a12int(primary_inputs:int|list, results:list):
    """transform the a1 complement to an unsigned integer

    Args:
        primary_inputs (int|list): the circuit primary inputs
        results (list): the a1 result from the logic gates operations

    Returns:
        results (list): a list of unsigned integers
    """    
    if isinstance(primary_inputs,list):
        nbits=len(primary_inputs)
    else:
        nbits=primary_inputs
    bit_mask=int('1'*2**nbits,2)
    for a,b in enumerate(results):
        if b<0:
            results[a]=b&bit_mask
    return results

def truth_table(primary_inputs:list, primary_outputs:list,circ_struc:list, show:str='y'):
    '''
    Params:
        primary_inputs(list): A list of the primary input signals index
        primary_outputs(list): A list of the primary output signals index
        circ_struct(list): A list with the relationships between inputs and outputs
    Returns:
        fault_free_circuit(dict): key:binary input for primary inputs value:fault-free output
    '''
    log("Beginning truth table generation")
    t0=datetime.now()
    b=bin2int(primary_inputs)
    normal_output=circuit_code(b,circ_struc)
    fault_free_circuit=a12int(primary_inputs,[normal_output[x] for x in primary_outputs])
    if show.lower()=='y':
        print("Printing fault free truth table")
        for x in primary_inputs: #printing PIs
            print(x, end="|")
        # print("|")
        for x in primary_outputs: #printing POs
            print(x, end="|")
        #printing values from fault-free value
        print()
        for x in range(2**len(primary_inputs)):
            for y in bin(x)[2:].zfill(len(primary_inputs)):
                print(y,end="|")
            for y in fault_free_circuit:
                print(bin(y)[2:].zfill(2**len(primary_inputs))[x],end="|")
            print()
    log("Finished generating truth table")
    print(f'Time take bit logic vector: {datetime.now()-t0}')
    return fault_free_circuit

def paths_generations(circ_struct:list, hierarchy:dict):
    """Generates the paths of signals withing a circuit

    Parameters
    --------
        circ_struct (list): the list of logic operations that relates inputs and outputs
        hierarchy (dict): the signal hierarchy of the circuit

    Returns
    --------
    line (list)
            A list with all the paths of the signals, all referenced to PIs
    """    
    log("Beggining path generation")
    t0=datetime.now()
    main_path=[[x] for x in hierarchy[0]]    
    last_item=[x for x in hierarchy[0]]
    fanout_path={}
    for g in circ_struct:
        output=g.split(' = ')[0]
        inputs=g.split(' = ')[1].split('(')[1][:-1].split(', ')
        for h in inputs:
            aux=last_item.count(h)
            if aux==0:
                if h not in hierarchy[0]: #if is a PI, disregard the fanout
                    fanout_path[h]=[]
                main_path.append([h, output])
                last_item.append(output)
            else:
                aux1=0
                while aux>0:
                    if h in last_item:
                        main_path[last_item.index(h,aux1)].append(output)
                        last_item[last_item.index(h,aux1)]=output
                        aux1+=1
                    aux-=1

    ##pending to fix, this function may result slow for large fanout data set. Think in implementing a 
    for g in fanout_path.keys():
        for h in main_path: #only analyze for the one in the PIs
            if g in h and h[0] in hierarchy[0]:
                fanout_path[g].append(h[0])#adding the PIs
    
    line={}
    for h in hierarchy.keys():
        if h>0:
            for i in hierarchy[h]:
                conn_PIs=[]
                if i in fanout_path.keys():# first check for the fanout branches
                    conn_PIs=fanout_path[i]
                else: # check for the main paths
                    for j in main_path:
                        if i in j:
                            if j[0] in hierarchy[0] and j[0] not in conn_PIs:
                                conn_PIs.append(j[0])
                            elif j[0] in fanout_path.keys():
                                for k in fanout_path[j[0]]:
                                    if k not in conn_PIs:
                                        conn_PIs.append(k)
                conn_PIs.sort()
                line[i]=conn_PIs
    log("Finished path generation")
    print(f"Time taken: {datetime.now()-t0}")
    return main_path,fanout_path,line

In [3]:
#reading the bench file to assign the input, output and intermediate connections.
#read only once (the rutine is the same for ALL the data)
def read_bench(bench_file:str):
    """Reads the bench file and determines the generation of truth table or paths generation

    Args
    -------
        bench_file : str
            the path of the bench file to evaluate

    Returns
    -------
        circ_struct: list
            relationship input, gate and output

        signal_hierarchy: dict
            signals map of the circuit
        
        truth table: dict
            ONLY if the circuit has 8 or less inputs, the truth table is generated
    """      
    t0=datetime.now()
    aux=0
    circ_struc=[] #circuit structure
    signal_hierarchy={} #key: level #value[list]: list of inputs. All signals are considered input
    signal_hierarchy[-1]=[]
    signal_hierarchy[0]=[] #[0]=PI [-1]=PO
    fanout_stems={}
    gate_inputs=0
    n_inv=0
    logic_gate_operations=0
    with open(bench_file) as f:
        while True:
            a=f.readline().strip()
            if aux>3:
                break
            if not(a):
                aux+=1
                continue
            else:
                if "#" in a:
                    continue
                else:
                    if 'INPUT' in a or 'OUTPUT' in a: #connections for primary input and primary output
                        b=a.split('(')[1][:-1]
                        if 'INPUT' in a:
                            signal_hierarchy[0].append(b)
                            fanout_stems[b]=0
                        else:
                            signal_hierarchy[-1].append(b)
                    else: # signal = gate (input signal)
                        circ_struc.append(a)
                        logic_gate_operations+=1
                        if 'NOT' in a: #getting the number of inverters
                            n_inv+=1

                        for z in a.split(' = ')[1].split('(')[1][:-1].split(','):
                            ind=0
                            while ind<(len(signal_hierarchy)-1):
                                if z in signal_hierarchy[ind]:
                                    ind+=1
                                    break
                                ind+=1
                            for x in signal_hierarchy.values():
                                if z in x:
                                    fanout_stems[z]+=1
                            if 'BUFF' not in a:
                                gate_inputs+=1
                        if ind not in signal_hierarchy:
                            signal_hierarchy[ind]=[]
                        signal_hierarchy[ind].append(a.split('=')[0].strip())
                        fanout_stems[a.split('=')[0].strip()]=0
    f.close()
    log (f"Finished reading bench file: {bench_file}")
    stems=0
    for y,z in fanout_stems.items():
        if z>1:
            stems+=1
    aux=0
    for z in signal_hierarchy.values():
        aux+=len(z)
    log (f'There area {aux*2} possible faulty circuits under the single-fault assumption')
    print(f'Number of PIs: {len(signal_hierarchy[0])}')
    print(f'Number of POs: {len(signal_hierarchy[-1])}')
    print(f'Number of fanout stems: {stems}')
    print(f'Number of gate inputs {gate_inputs}')
    print(f'Number of inverters: {n_inv}')
    print(f'Number of logic gate operations: {logic_gate_operations}')
    print(f'Number of collapsed faults: {2*(len(signal_hierarchy[-1])+stems)+gate_inputs-n_inv}')

    print(f'time taken: {datetime.now()-t0}')
    if len(signal_hierarchy[0])<8: #it was chosen 8, in order to use an 8 switch input from wokwi
        log(f'circuit is small enough to perform a functional testing, generating truth table')
        z=truth_table(signal_hierarchy[0],signal_hierarchy[-1],circ_struc)
        return circ_struc, signal_hierarchy,z
    else:
        log(f'circuit has over 10  inputs, too large to perfom a functional testing, a list with line justification is returned instead')
        # z=paths_generations(circ_struc,signal_hierarchy)
        return circ_struc, signal_hierarchy

In [ ]:
def test(circ_struc: list,signals:dict, test_file: str="", test_vectors:list=[]):
    """reads the test vector form the test file or provided from another function and print the fault yield

    Args:
        circ_struc (list): Logic gates operations
        signals (dict): all the signals within the circuit
        test_file (str, optional): file with test vectors to implement. Defaults to "".
        test_vectors (list, optional): test vectors received from another function. Defaults to [].
    """    
    t0=datetime.now()
    primary_input=signals[0]
    primary_output=signals[-1]
    signal_line={x:[] for x in primary_input}
    aux=0
    if test_file: #test file is provided
        print("reading from test file")
        with open(test_file) as f:
            while True:
                line=f.readline().strip()
                if not(line):
                    break
                else: #I assume the data is organized in the bench stated order
                    [signal_line[y].append(line[x]) for x,y in enumerate(primary_input)]
                    aux+=1
        f.close()
    elif test_vectors: #if test vector provided from ATPG
        print("testing from ATPG results")
        for a in test_vectors:
            [signal_line[y].append(a[x]) for x,y in enumerate(primary_input)]
            aux+=1
    
    for x in signal_line:
        signal_line[x]=int(''.join(signal_line[x]),2)
    fault_free=circuit_code(signal_line,circ_struc)#fault free
    aux3={x:y for x,y in fault_free.items() if x in primary_output} #only work with POs
    aux1=0
    aux2=0
    for u in signals.values():
        for x in u:            
            for y in ['0','1']:                
                SA_fault=circuit_code(signal_line,circ_struc,{x:int(y*aux,2)})#faulty circuit
                for z in primary_output:
                    if aux3[z]!=SA_fault[z]:
                        aux2+=1                    
                        break
                aux1+=1
    ##information to show to user
    print('*'*100)
    print(f"Detected faults: {aux2}")
    print(f'Total # of faults: {aux1}')
    print(f'Detection percentage: {aux2/aux1*100}')
    print(f'Time taken: {datetime.now()-t0}')
    print('*'*100)
    log(f"Finished running test file {test_file}")
    return

Code to draw the circuit in the Wokwi

In [ ]:
#logic gates codes to draw circuits in wokwi
def logic_gate_diagram(gate:str,number:int,top:int,left:int):
    '''
    Params
    -------
        gate(str): the logic gate desired
        number(int): the number for id
        top(int): top value for wokwi reference system
        left(int): left value for wokwi reference system
    Returns
    -------
        part_json(str): the corresponding json to draw the circuit on Wokwi
        output_json(str): the correspondin json to indicate the ouput\
    '''
    if gate=='NAND':
        type="wokwi-gate-nand-2"
    elif gate=='AND':
        type="wokwi-gate-and-2"
    elif gate=='NOR':
        type= "wokwi-gate-nor-2"
    elif gate=='OR':
        type="wokwi-gate-or-2"
    elif gate=='XOR':
        type="wokwi-gate-xor-2"
    elif gate=='XNOR':
        type="wokwi-gate-xnor-2"
    elif gate=='NOT':
        type="wokwi-gate-not"
    elif gate=='BUFF':
        type="wokwi-gate-buffer"
        
    return f'"type": "{type}", "id":"{gate.lower()}{str(number)}","top":{top},"left":{left}'

def gate_left_top(circ_struc:list,hierarchy:dict):
    """Determines the top and left coordinates of the logic gates

    Args:
        circ_struc (list): _description_
        hierarchy (dict): _description_

    Returns:
        rel_pos (dict): A dictionary with the following structure 
            Key:signal
            Value (list):[left coordinate, top coordinate]
    """
    rel_pos={} #key: signal #value[left,top]
    #####new code
    for y,z in hierarchy.items():
        if y>=0:
            for u,x in enumerate(z):
                rel_pos[x]=[y,u]

    for s in circ_struc:
        #logic gates
        aux=s.split('=')[0].strip()#output
        aux1=s.split('=')[1].split('(')[1][:-1].split(',')#inputs
        w=0      
        for x in aux1:
            w+=rel_pos[x.strip()][1]
        rel_pos[aux][1]+=(w/len(aux1))

    return rel_pos

def lines(gates:dict,circ, connections:list=[]):
    '''creates the lines conecting the logic gates
    gates=the Key: signal, value:[gateid, color]
    circ= the relationship between the signals
    '''
    aux1=['A','B']
    g_out=circ.split(' = ')[0]#output
    g_in=circ.split(' = ')[1].split('(')[1][:-1].split(',') #inputs
    if g_out in gates:
        for x in g_in:
            if len(g_in)>1: #2 signals inputs
                aux3=aux1[g_in.index(x)]
            else:
                aux3='IN'
            aux4=''
            if 'sw'not in gates[x.strip()][0]:
                aux4=':OUT'
            # print(f'["{gates[x.strip()][0]}{aux4}","{gates[g_out][0]}:{aux3}","{gates[x.strip()][1]}"]')#prints logic gate connection and color
            connections.append(f'["{gates[x.strip()][0]}{aux4}","{gates[g_out][0]}:{aux3}","{gates[x.strip()][1]}"]')#prints logic gate connection and color
    return connections

In [ ]:
a,b,c=read_bench('./data.nogit/c3.bench')
rel_pos=gate_left_top(a,b)
parts=[] #for logic gates and text labels
connections=[] #for wires

In [ ]:
import random
aux={}#key:signal value=[gate, color]

for y,z in b.items():#establishing the inputs connection with the PIs and signals coloring
    if y>=0:
        aux1=1
        for x in z:
            if y==0:
                aux[x]=['sw1:'+str(aux1)+'b']
                aux1+=1
            else:
                aux[x]=[]
            #generating color for the signal
            hex_color_code='#'
            for _ in range(6):
                hex_color_code+=random.choice('0123456789abcdef')
            aux[x].append(hex_color_code)

for d,e in enumerate(a): # for logic gates
    gate=e.split(' = ')[1].split('(')[0]
    g_out=e.split(' = ')[0]
    aux[g_out].insert(0,gate.lower()+str(d))

    parts.append(f'"type": "wokwi-text","id":"text{str(d)}","top":{rel_pos[g_out][1]*100},"left":{rel_pos[g_out][0]*120+100},"attrs":{'{'}"text":"{g_out}"{'}'}') #text labels
    parts.append(logic_gate_diagram(gate,d,rel_pos[g_out][1]*100,rel_pos[g_out][0]*120))#logic gate
    connections=lines(aux,e,connections)
print('"parts":[')

for z in parts:
    print("{",end="")
    print(z,end="")
    print("},")
print('{"type": "wokwi-dip-switch-8", "id": "sw1", "top": 0, "left": 0, "rotate": 90}')#adding a switch for the ipnuts
print("],")
print('"connections":[')
for z in connections:
    print(z,',')
print("]")
print(f'Try out the circuit in the following website: https://wokwi.com/projects/354858054593504257')

In [ ]:
wokwi_diagram_json={
    "version":1,
    "author":"Anonymous maker",
    "editor":"wokwi",
    "parts":[], #logic gates and text
    "connnections":[],#wires between logic gates
    "dependencies":{}
}

Stuck-at fault code
Fault equivalency simulations for every logic gate

For any gate try SA0 and SA1 at both input and output

In [ ]:
#for each logic gate testing fault colapsing
def brute_SA_faults_logic_gates_colapsing(gate:str,inputs:str|int):
    '''
    Params:
        gate(str): the logic gate to evaluate
        inputs(str|int): is the number of inputs for the gate str:'0000' int:2
    Returns:
    '''

    # inputs=2
    # gate='NAND'
    if isinstance(inputs,int):
        lenght=inputs
    elif isinstance(inputs,str):
        lenght=len(inputs)
    else:
        # return "the provided input is not in a valid format"
        print("the provided input is not in a valid format")

    test='SA0'
    failure=''
    signal=['A','B','Output']
    #SA for the inputs
    for a in range(2**lenght+2):
        for b in range(2**lenght):
            pi_input=bin(b)[2:].zfill(lenght)
            po=logic_gate(gate,list(pi_input))
            sa_input=pi_input[0:(a//2)]+str(int(test[-1]))+pi_input[(a//2)+1:]
            if a<2**lenght:
                sa_output=logic_gate(gate,list(sa_input))
            else:
                sa_input='xx'
                sa_output=int(test[-1])
            
            if sa_output!=po:
                failure="failed"
            else:
                failure=''
            print(f'|{gate}|{pi_input}|{test}[{signal[a//2]}]|{sa_input}|{po}|{sa_output}|{failure}|')
        if test=='SA0':
            test='SA1'
        else:
            test='SA0'

In [ ]:
#for perfoming brute SA fault 
def functional_SA_single_faults(circuit:list, hierarchy:dict):
    """performs an Stuck-at 0 and Stuck-at 1 single fault for every circuit segment
    Compares the obtained result against the truth table

    Args:
        circuit (list): relationship between logic gate, input and output
        hierarchy (dict): signal map of the circuit

    Returns:
        SA_signal_line (dict): key:signal_SA0/SA1 value:[list of output]
    """    
    log("Generating SA fault truth table")
    lenght=[]
    test_vector={}
    SA_signal_line={}
    for z in hierarchy.values():
        for y in z:
            lenght.append(y)
    #SA for the inputs
    for a in lenght:
        for test in ['SA0','SA1']:
            SA_signal_line[f'{a}_{test}']=[]
            test_vector=bin2int(hierarchy[0])
            outputs=circuit_code(test_vector,circuit,{a:int(test[-1]*2**len(hierarchy[0]),2)})
            SA_signal_line[f'{a}_{test}']=a12int(hierarchy[0],[outputs[x] for x in hierarchy[-1]])
    log("Finished generating SA fault truth table")
    return SA_signal_line
def comparte_truth_SA_tables(fault_free_truth_table:list, SA_truth_table:dict, hierarchy:dict, show:str ='y',file:str='null'):
    """compares the results from the fault free and SA fault truth table

    Args
    -------
        fault_free_truth_table (list): fault free truth table results. It's assumed that the index of the list corresponds to the binary input
        SA_truth_table (dict): SA fault truth table result. Key:(signal_SA0/SA1) Value (list):result of SA fault truth table
        hierarchy (dict): Circuit signal hierarchy
        show (str): to print result on screen
        file (str): name of the file to generate readme file in github

    Returns
    -------
        SA_vector (dict): 
            Table with the SA faults and test vector that generate failures at the primary outputs.
            Key: SA fault
            Value: test vector
        
        test_vector (dict):
            Table with the test vector and SA faults that generate failures at the primary outputs
            Key: test vector
            Value: SA fault
        
    """    
    log("Comparing fault free and functional SA fault truth tables")

    SA_vector={}
    test_vector={x:[] for x in range(2**len(hierarchy[0]))}
    length=0
    for d in SA_truth_table.keys():
        aux=0
        aux2=0
        SA_vector[d]=[]
        while aux<len(fault_free_truth_table):
            if fault_free_truth_table[aux]!=SA_truth_table[d][aux]:#vector comparison
                aux1=fault_free_truth_table[aux]^SA_truth_table[d][aux] #finding the different values
                aux2=aux1|aux2 #grouping per fault
                # print(aux2)
            aux+=1
        SA_vector[d]=aux2
        [test_vector[x].append(d) for x,y in enumerate(bin(aux2)[2:].zfill(2**len(hierarchy[0]))) if y=='1']
        # print(d,aux2, "vector: ",[bin(x)[2:].zfill(len(hierarchy[0])) for x,y in enumerate(bin(aux2)[2:].zfill(2**len(hierarchy[0]))) if y=='1'])
        print(d,aux2, "vector: ",[x for x,y in enumerate(bin(aux2)[2:].zfill(2**len(hierarchy[0]))) if y=='1'])
    [length:=max(length,len(y)) for y in test_vector.values()]
    print()
    while length>0:
        print(f'{length} detected SA faults by test vectors: ',end=' ')
        for aux1, aux2 in test_vector.items():
            if len(aux2)==length:
                print(bin(aux1)[2:].zfill(len(hierarchy[0])), end='|')
        print()
        length-=1
    
    if file!='null':
        #format data table
        with open(f'{file}.md','w') as f:
            f.write('| SA fault | ')
            for u in range(w):
                f.write('test vector | ')
            f.write('\n')
            f.write ('| :---: | ')
            for u in range(w):
                f.write(' :---: | ')
            f.write('\n')
            while w>0:
                for y,z in SA_vector.items():
                    if len(z)==w:
                        f.write ('|')
                        f.write(f'{y} |')
                        for x in z:
                            f.write(f'{x}|')
                        f.write('\n')
                w-=1
            f.write('\n\n\n')

            f.write('| Test vector | ')
            for u in range(v):
                f.write('SA fault | ')
            f.write('\n')
            f.write ('| :---: | ')
            for u in range(v):
                f.write(':---: | ')
            f.write('\n')
            
            while v>0:
                for y,z in test_vector.items():
                    if len(z)==v:
                        f.write ('|')
                        f.write(f'{y}|')
                        for x in z:
                            f.write(f'{x}|')
                        f.write('\n')
                v-=1
        f.close()
        
    log("Finished comparing fault free and functional SA fault truth tables")
    return SA_vector, test_vector

In [ ]:
circ_struct,hierarchy,main_paths=read_bench('./data.nogit/c17.bench')
# a,b,c=read_bench('./kelvin_testing/c17_AND.bench')
print()
fanout_path=functional_SA_single_faults(circ_struct,hierarchy)
print()
e,f=comparte_truth_SA_tables(main_paths,fanout_path,hierarchy)

In [ ]:
import itertools
import math

In [ ]:
#def pareto():
max_test_vector=math.ceil(0.2*len(f.keys()))
vector_to_test=[]
best_test_vector=[]
per=0
#ordering test vector from most to least detected SA faults
aux1=0
for y,z in f.items():
    aux1=max(len(z),aux1)

while (aux1>0 and len(vector_to_test)<max_test_vector):
    for y,z in f.items():
        if len(z)==aux1:
            vector_to_test.append(y)
    aux1-=1
print("Pareto test vector: ",max_test_vector)
#doing combination to determine the percentage of covering
aux3=0
while aux3<len(vector_to_test):
# for g in itertools.combinations(f.keys(),math.ceil(0.2*len(f.keys()))):
    per=0
    for g in itertools.combinations(vector_to_test,aux3):
        det_sa_faults=[]
        for h in g:
            aux=0
            while aux<len(f[h]):
                if f[h][aux] not in det_sa_faults:
                    det_sa_faults.append(f[h][aux])
                aux+=1
        aux2=len(det_sa_faults)/len(e.keys())
        # print(g,aux2)
        if aux2==1:
            best_test_vector=g
            per=aux2
            break
        elif aux2>per:
            best_test_vector=g
            per=aux2
    print(best_test_vector,per)
    aux3+=1

ATPG
getting paths

Tseitin Transform

In [3]:
from pysat.formula import CNF
from pysat.solvers import Solver

def solve(clauses):
    with Solver(bootstrap_with=CNF(from_clauses=clauses)) as solver:
        if solver.solve():
            return solver.get_model()
        return []
def gen_tseitin(circ_struc:list):
    """Generates the tseitin formulation for circuit.
    It only works for 2 input logic gates

    Params
    ------
        circ_struc:str
            the logic gates with input and outputs

    Returns
    -------
        tseitin:list
            the tseitin equation for the given circuit
        num_notation: dict|str
            a numeric auxiliary notations only for those circuit that has lettered signals
    """    
    tseitin=[] #setting up the tseitin equations
    num_notation={} #number notation for circuits with lettered signals
    aux1=1 
    for h in circ_struc:
        aux2,gate,ins=parts(h)
        
        if aux2 not in num_notation:
            num_notation[aux2]=aux1
            output=aux1 #output in holder numerical value
            aux1+=1
        else:
            output=num_notation[aux2]
        inputs=[]
        for z in ins:
            if z not in num_notation:
                num_notation[z]=aux1
                inputs.append(aux1)
                aux1+=1
            else:
                inputs.append(num_notation[z])


        #for negated use '-' negative sign
        if gate.upper()=='NAND': 
            tseitin.append([inputs[0], output])
            tseitin.append([inputs[1],output])
            tseitin.append([-inputs[0],-inputs[1],-output])
        elif gate.upper()=="AND":
            tseitin.append([inputs[0],-output])
            tseitin.append([inputs[1],-output])
            tseitin.append([-inputs[0],-inputs[1],output])
        elif gate.upper()=="OR": 
            tseitin.append([-inputs[0], output])
            tseitin.append([-inputs[1],output])
            tseitin.append([inputs[0],inputs[1],-output])
        elif gate.upper()=="NOT":
            tseitin.append([-inputs[0], -output])
            tseitin.append([inputs[0],output])
        elif gate.upper()=="BUFF":
            tseitin.append([-inputs[0], output])
            tseitin.append([inputs[0],-output])
        elif gate.upper()=="NOR": 
            tseitin.append([-inputs[0], -output])
            tseitin.append([-inputs[1],-output])
            tseitin.append([inputs[0],inputs[1],output])
        elif gate.upper()=="XOR": 
            tseitin.append([-inputs[0],-inputs[1],-output])
            tseitin.append([inputs[0],inputs[1],-output])
            tseitin.append([inputs[0],-inputs[1],output])
            tseitin.append([-inputs[0],inputs[1],output])
        elif gate.upper()=="XNOR": 
            tseitin.append([-inputs[0],-inputs[1],output])
            tseitin.append([inputs[0],inputs[1],output])
            tseitin.append([inputs[0],-inputs[1],-output])
            tseitin.append([-inputs[0],inputs[1],-output])
        
    if len(num_notation)==0:
        num_notation='No auxiliary numeric notation necessary'
    
    return tseitin,num_notation


In [8]:
a,b,c,d=new_read_bench('./data.nogit/c17.bench')
e,f=gen_tseitin(a)

# 00028.065 - Finished reading bench file
****************************************************************************************************
# of POs:  2
# of fanout stems:  3
# of gate inputs:  9
All possible faults:  34
Calculated collapsed faults:  22
Time taken: 00000.000
****************************************************************************************************


Using SAT solver for solving the tseitin equation

In [ ]:
assignments=[[-3]] #forces variable 7=0 (low)
model=solve(c+assignments)

print(model)
if len(d)>0:
    aux=list(d.keys())
    for y,z in enumerate(model):
        if z>0:
            aux1=1
        else:
            aux1=0
        print(aux[y],aux1)


PODEM Line Justification

Consider 
1. Signal hierarchy has the signal ordered by "level" so a level 5 signal will NEVER have a dependency on level 5 or higher
2. circ structure has the relationship between logic gates, inputs and outputs
3. path has a list or directly relationship. (pending to fix fanout paths). This approach is faster than looking the signal hierarchy because it only looks on the signal on the same path. While the signal hierarchy will have signal on different paths but same level.


In [ ]:
#def PODEM(hierarchy:dict, main_paths:list, fanout_paths:dict):
def line_justification(hierarchy:dict, main_paths:list, fanout_paths:dict):
    ## line justification
    line={}
    for h in hierarchy.keys():
        if h>0:
            for i in hierarchy[h]:
                conn_PIs=[]
                if i in fanout_paths.keys():# first check for the fanout branches
                    conn_PIs=fanout_paths[i]
                else: # check for the main paths
                    for j in main_paths:
                        if i in j:
                            if j[0] in hierarchy[0] and j[0] not in conn_PIs:
                                conn_PIs.append(j[0])
                            elif j[0] in fanout_paths.keys():
                                for k in fanout_paths[j[0]]:
                                    if k not in conn_PIs:
                                        conn_PIs.append(k)
                line[i]=conn_PIs
    return line

Justification

In [ ]:
def justification(circ,signal,value:str ='0',needed_PIs:dict={}):
    mid_signals=[signal]
    circ.sort(reverse=True)
    for y in circ:
        output_signal=y.split(' = ')[0]
        if output_signal in mid_signals:
            [mid_signals.append(x) for x in y.split(' = ')[1].split('(')[1][:-1].split(', ') if x not in mid_signals]
    mid_signals.sort()
    mid_circ=[]
    for x in a:
        #only check the output of the logic gate
        v=x.split(' = ')[0].strip()
        if v in mid_signals:
            if x not in mid_circ:
                mid_circ.append(x)
    mid_circ.sort(reverse=True)

    testing_signal_values={} #key=siganl #value=expected value (0/1)&(weak[can change]/strong[cant change])
    testing_signal_values[signal]=value+'s'
    for operation in mid_circ:
        output_signal=operation.split(' = ')[0]
        gate=operation.split(' = ')[1].split('(')[0]
        input_signal=operation.split(' = ')[1].split('(')[1][:-1].split(', ')
        value=testing_signal_values[output_signal]

        #analyze, what happens when the previous signal is 'w'??
        if gate.upper()=='AND':
            if value[0]=='1':
                to_assign_value='1s' #value can NOT change
            else:
                to_assign_value='0w' #value can change

        elif gate.upper()=='NAND': # or gate.upper()=='XOR' 
            if value[0]=='0':
                to_assign_value='1s' #value can NOT change
            else:
                to_assign_value='0w' #value can change

        elif gate.upper()=='OR':
            if value[0]=='0':
                to_assign_value='0s' #value can NOT change
            else:
                to_assign_value='1w' #value can change
        
        elif gate.upper()=='NOR':
            if value[0]=='0':
                to_assign_value='1s' #value can NOT change
            else:
                to_assign_value='0w' #value can change

        elif gate.upper()=='XOR' or gate.upper()=='XNOR': #pending to test, because all inputs in this gate must be equals
            if value[0]=='0':
                to_assign_value='0s' #value can NOT change?
            else:
                to_assign_value='1s' #value can change?

        elif gate.upper()=='NOT':
            if value[0]=='1':
                to_assign_value='0'
            else:
                to_assign_value='1'
            to_assign_value+=value[1] #the weak and strong is transfered 
        
        elif gate.upper()=='BUFF':
            if value[0]=='0':
                to_assign_value='0'
            else:
                to_assign_value='1'
            to_assign_value+=value[1] #the weak and strong is transfered 

        for x in input_signal:
            if x in needed_PIs and to_assign_value!=needed_PIs[x] and needed_PIs[x]!='X':
                print("This input is being forced from a previous line justification")
                #try to change the weak signal 
                for y,z in testing_signal_values.items():
                    print(f'Signal:{y},Value:{z}')
            else:
            # if testing_signal_values[x][1]=='w': #pending to analyze when to 's' intervene
                testing_signal_values[x]=to_assign_value+value[1] #it adds the information from the output to consider
    return testing_signal_values

In [ ]:
implications={
    "AND":['000','010','0X0','0D0','0E0','111','1XX','1DD','1EE','XXX','XDX','XEX','DDD','DE0','EEE'], #input(a)input(b)output
    "NAND":['001','011','0X1','0D1','0E1','110','1XX','1DE','1ED','XXX','XDX','XEX','DDE','DE1','EED'],
    "OR":['000','011','0XX','0DD','0EE','111','1X1','1D1','1E1','XXX','XDX','XEX','DDD','DE1','EEE'],
    "NOR":['001','010','0XX','0DE','0ED','110','1X0','1D0','1E0','XXX','XDX','XEX','DDE','DE0','EED'],
    "XOR":['000','011','0XX','0DD','0EE','110','1XX','1DE','1ED','XXX','XDX','XEX','DD0','DE1','EE0'],
    "XNOR":['001','010','0XX','0DE','0ED','111','1XX','1DD','1EE','XXX','XDX','XEX','DD1','DE0','EE1'],
    "NOT":['0-1','1-0','X-X','D-E','E-D'],
    "BUFF":['0-0','1-1','X-X','D-D','E-E']
}

Propagation

In [ ]:
def propagation(circuit,signal,value):
    desired_signal=[]
    desired_signal.append(signal)
    circuit.sort()
    
    mid_circ=[]
    for x in circuit:
        #only check the input of the logic gate
        v=x.split(' = ')[1].split('(')[1][:-1].split(', ') #is a list
        aux=0
        while aux<len(v):
            if v[aux] in desired_signal:
                mid_circ.append(x)
                desired_signal.append(x.split(' = ')[0].strip()) #get the output for the next             
                break
            aux+=1
    
    testing_signal_values={} #key=siganl #value=expected value (0/1)&(weak[can change]/strong[cant change])
    testing_signal_values[signal]=value
    pending_testing={}#needed signals to test in order to propagate the fault
    
    for operation in mid_circ:
        output_signal=operation.split(' = ')[0]
        gate=operation.split(' = ')[1].split('(')[0]
        input_signal=operation.split(' = ')[1].split('(')[1][:-1].split(', ')

        for z in input_signal:
                if z in testing_signal_values:
                    value=testing_signal_values[z]
                for y in implications[gate]:
                    if value in y[:-1] and y[-1] in ['D','E']:
                        next_output=y[-1]
                        if z!=signal and z not in testing_signal_values: #prevents multiple assignment in fanout circuits
                            if z not in pending_testing:# and z not in testing_signal_values: #for testing the other input, this information will need to have line justification
                                pending_testing[z]=[]
                            for x in y[:-1]:
                                if x not in pending_testing[z] and x not in ['D','E']:# and z not in testing_signal_values: #this ensures a single stuck at fault
                                    pending_testing[z].append(x)
                        else:
                            continue
        testing_signal_values[output_signal]=next_output#assign the corresponding SA fault propagation to the output signal 
        signal=output_signal

    # print(testing_signal_values)
    # print(pending_testing)
    return pending_testing 

In [ ]:
def singleSA (circuit, hierarchy , signal:str, SA_value:str='0'):
    PIs=dict.fromkeys(hierarchy[0],'X')
    if SA_value=='0':
        z=propagation(circuit, signal, 'D')
        aux='1'
    elif SA_value=='1':
        z=propagation(circuit, signal, 'E')
        aux='0'
    
    #first testing the solicited signal 
    y=justification(circuit, signal,aux)
    print(f"PIs that activate SA@{SA_value} in signal:{signal}")
    for x in PIs:
        if x in y:
            PIs[x]=y[x]
    print(PIs)
    print(f'Pending to test: {z}')
    
    #then run the other needed testing obtained from the propagation circuit
    pending_testing=z#needed signals to test in order to propagate the fault
    
    test_vector=[]
    #aggregate the posibilities from the other signals to be tested
    for y,z in pending_testing.items():
        for x in z:
            u=justification(circuit,y,x,PIs)
            print(u)
            aux=''
            for t,s in PIs.items():
                if s =='X': #looking to replace 
                    if t in u:
                        print(f'{t}:{u[t][0]}',end=" ")
                        aux+=u[t][0]
                    else:
                        print(f'{t}:X',end=" ")
                        aux+='X'
                else:
                    print(f'{t}:{s[0]}',end=" ")
                    aux+=s[0]
            print(aux)
            test_vector.append(aux)
            print()
    return test_vector

In [ ]:
### Testing
# a,b,c=read_bench('./kelvin_testing/c1.bench')
# a,b,c=read_bench('./kelvin_testing/c17_XNOR.bench')
a,b,c=read_bench('./data.nogit/c3.bench')
# a,b,c=read_bench('./data.nogit/c17.bench')
# c,d,e=paths_generations(a,b)
# a,b,c=read_bench('./kelvin_testing/c17_XOR.bench')
# a,b,c=read_bench('./data.nogit/c17.bench')
test_vector=[]
for x,y in b.items():
    if x>=0:
        for u in y:
            for t in ['0','1']:
                z=singleSA(a,b,u,t)
                for t in z:
                    if t not in test_vector:
                        test_vector.append(t)
                print()
            print('*'*100)
print("For this circuit, this vectors may detect all SA faults")
print("Total: ",len(test_vector))
for z in test_vector:
    print(z)

In [ ]:
vectors={}
for d,e in enumerate(c.keys()):
    # for i in ['0','1']:
    #     f=find_test(a,b,c,e,i)
    #     for g in f:
    #         if g not in vectors:
    #             vectors[g]=0
    #         vectors[g]+=1
    print(e,type(e))
print(vectors)

In [ ]:
print('AND Gate: 0X',logic_gate('AND',['0','X']))
print('AND Gate: 1X',logic_gate('AND',['1','X']))

Matrix operation

In [ ]:
#to generate the truth table 
import itertools as it
import numpy as np
def mat(pi:int):
    z=np.array(list(it.product('01',repeat=pi)))
    return z

In [ ]:
y=mat(10)
print(y)

In [ ]:
bin2int([1,2,3,6,7])

In [ ]:
#comparing timing functions (matrix generation vs int PPSFP)

import timeit
aux=1
while aux<15:
    aux2=[x for x in range(aux)]
    # Time function_one
    time_one = timeit.timeit("mat(aux)", globals=globals(),number=10000)

    # Time function_two
    time_two = timeit.timeit("bin2int(aux2)", globals=globals(), number=10000)
    print(f"Number of inputs: {aux}")
    print(f"Time taken by matrix generation: \t{time_one:.6f} seconds")
    print(f"Time taken by integer vector: \t{time_two:.6f} seconds")

    if time_one < time_two:
        print("function_one is faster.")    
    elif time_two < time_one:
        print("function_two is faster.")
    else:
        print("Both functions took approximately the same time.")
    aux+=1
    print('*'*1000)


In [3]:
# import time
# from datetime import datetime
t0 = time.perf_counter()

def log(msg: str):
    dt = time.perf_counter() - t0

    print(f"# {dt:09.3f} - {msg}")

def parts(se:str):
    output=se.split("=")[0].strip()
    gate=se.split("=")[1].split("(")[0].strip()
    inputs=[]
    for b in se.split("=")[1].split("(")[1][:-1].split(","):
        inputs.append(b.strip())
    return output, gate, inputs

#ordering circuit gates
def topo_order(gates, inputs):
    known, order, seen = set(inputs), [], set()
    while len(seen) < len(gates):
        progressed = False
        for z in gates:
            out, gt, ins = parts(z)
            if out in seen: continue
            if all(u in known for u in ins):
                order.append(z)
                # order.append((out, gt, ins))
                seen.add(out); known.add(out); progressed = True
        if not progressed:
            rem = [o for o,_,_ in gates if o not in seen]
            raise RuntimeError(f"Not previously seen signal：{rem}")
    return order

def col_fault(circuit, POs):
    collapsed_faults=[]
    equivalences=[]
    not_gates=[]
    gate_inputs=0
    
    for a in reversed(circuit):
        b,c,d=parts(a)
        if c!="BUFF": gate_inputs+=len(d)
        if b in POs:
            for e in ["@0","@1"]:
                collapsed_faults.append(b+e)
        match c:
            case "AND":
                [collapsed_faults.append(e+"@1") for e in d]
                [equivalences.append(e+"@0") for e in d]
            case "OR":
                [collapsed_faults.append(e+"@0") for e in d]
                [equivalences.append(e+"@1") for e in d]
            case "NAND":
                [collapsed_faults.append(e+"@1") for e in d]
                [equivalences.append(e+"@0") for e in d]
            case "NOR":
                [collapsed_faults.append(e+"@0") for e in d]
                [equivalences.append(e+"@1") for e in d]
            case _: #XOR, XNOR, BUFF, NOT
                if c=="NOT":
                    not_gates.append(a)
                for e in d:
                    for f in ["@0","@1"]:
                        collapsed_faults.append(e+f)
    for a in not_gates:
        b,c,d=parts(a)
        for e in ["@0","@1"]:
            if b+e in equivalences:
                for f in d:
                    collapsed_faults.remove(f+"@"+str(abs(1-int(e[-1]))))
            if b+e in collapsed_faults:
                for f in d:
                    collapsed_faults.remove(f+"@"+str(abs(1-int(e[-1]))))

    collapsed_faults=list(sorted(set(collapsed_faults)))
    return collapsed_faults

def new_buffers(circuit, signals, POs):
    aux={}
    buff_ope=0
    fanout_stems=0
    #buffers insertion and filtering
    for a,b in signals.items():
        if len(b)>1: #buffer insertion
            fanout_stems+=1
            for d,c in enumerate(b):
                if "BUFF" not in c:
                    if c in circuit:
                        position=circuit.index(c)
                        mod_gate=c
                    else:
                        for e in circuit:
                            if a in e:
                                f,g,h=parts(e)
                                i=f+" = "+g
                                if i in c:
                                    position=circuit.index(e)
                                    mod_gate=e
                    new_gate=a+'.b'+str(d)+"=BUFF("+a+")"
                    _,_,inputs=parts(mod_gate)
                    inputs[inputs.index(a)]=a+".b"+str(d)
                    old_gate=mod_gate[:mod_gate.index("(")]+"("+",".join(inputs)+")"
                    circuit[position]=old_gate
                    circuit.insert(position,new_gate)
                    buff_ope+=1
                    #on signals
                    b[d]=new_gate
                    if a+".b"+str(d) not in aux:
                        aux[a+".b"+str(d)]=[]
                    aux[a+".b"+str(d)].append(old_gate)
        else:
            if "BUFF" in b[0]: #remove one buffer
                if b[0] in circuit:
                    o,_,i=parts(b[0])
                    circuit.remove(b[0])
                    if o in POs:
                        pos=POs.index(o)
                        POs.insert(pos,i[0])
                        POs.remove(o)
    signals.update(aux)
    return circuit, signals

def circuit(circuit, initial_values, PIs, POs, SA={}):
    run={y:initial_values[x] if y not in SA else SA[y] for x,y in enumerate(PIs)}
    for a in circuit: 
        o,g,i=parts(a)
        if o in SA:
            run[o]=SA[o]
        else:
            run[o]=logic_gate(g,[run[b] for b in i])
    results=[run[x] for x in POs]
    return results

In [4]:
# import time
def new_read_bench(bench_file):
    t0=time.perf_counter()
    circuit, PIs, POs,=[],[],[]
    signals={}#key: signal, value:[logic gate signal input]

    #aux variables
    fanout_stems=0
    gate_inputs=0

    # def new_buffers(file):
    with open(bench_file) as a:
        #reading file logic gates
        for b in a.readlines():
            # if "#" not in b and "INPUT" not in b and "OUTPUT" not in b and len(b)>1:
            if "=" in b:
                circuit.append(b.strip())
                outputs,_,inputs=parts(b.strip())
                if outputs not in signals: POs.append(outputs)
                if outputs in PIs: PIs.remove(outputs)
                for c in inputs:
                    if c not in POs and c not in PIs and c not in signals:
                        PIs.append(c)
                    if c in POs:
                        POs.remove(c)
                    if c not in signals:
                        signals[c]=[]
                    signals[c].append(b.strip())
    a.close()
    print("Finished reading bench file: ", bench_file)
    log("Finished reading bench file")
    #order the circuit
    circuit=topo_order(circuit,PIs)
    
    #insert and delete redundant buffers
    circuit, signals=new_buffers(circuit, signals, POs)
    all_signals=[x for x in signals]

    #get all collapsed faults for SA faults
    collapsed_faults=col_fault(circuit, POs)

    for a,b in signals.items():
        if len(b)>1:
            fanout_stems+=1
        if ".b" not in a:
            gate_inputs+=1

    all_possible_faults=len(signals)*2+len(POs)*2
    print("*"*100)
    print("# of POs: ",len(POs))
    print("# of fanout stems: ",fanout_stems)
    print("# of gate inputs: ",gate_inputs)
    # print("# of inverter gates: ",len(not_gates))
    print("All possible faults: ",all_possible_faults)
    # print("Expected collapsed faults: ",2*(len(POs)+fanout_stems)+gate_inputs-len(not_gates))
    print("Calculated collapsed faults: ",len(collapsed_faults))
    print(f"Time taken: {(time.perf_counter()-t0):09.3f}")
    print("*"*100)
    return circuit,PIs, POs, collapsed_faults, all_signals

In [5]:
# from datetime import datetime
def test1(test_file):
    t0=datetime.now()
    #reading from test file
    matrix=[a for a in open(test_file).read().split() if a]
    s0=int('0'*len(matrix),2)
    s1=int('1'*len(matrix),2)
    packed = []
    for u in range(len(matrix[0])):
        value = 0
        for i, row in enumerate(matrix):
            if row[u] == '1':
                value |= (1 << i)
        packed.append(value)
    # log("Finished reading test file and generating SA0 and SA1 values")
    print("Time taken: ",datetime.now()-t0)
    return packed, s0,s1

In [ ]:
import pandas as pd
def test2(test_file):
    t0=datetime.now()
    a=pd.read_csv(test_file, header=None)
    matrix=a[0].apply(list).apply(pd.Series)
    matrix = [int(''.join([str(x[u]) if x[u]!='-' else '0' for x in matrix]),2) for u in range(len(matrix[0]))]
    print("Time taken: ",datetime.now()-t0)
    return matrix

In [8]:
# import os
tests_files=[a.split(".")[0] for a in os.listdir('./data.nogit/') if 'test' in a]

for aux in tests_files:
    a,b,c,d,e=new_read_bench(f"./data.nogit/{aux}.bench")
    z,x0,x1=test1(f"./data.nogit/{aux}.tests")
    print("# of PIS: ",len(b))
    print("# of POS: ",len(c))
    detected=0
    all_faults=0
    ffc=circuit(a,z,b,c) #fault free circuit
    print()
    t1=time.perf_counter()
    for f in e:        
        for g in [x0, x1]:
            aux1={f:g}
            res=circuit(a,z,b,c,aux1)
            all_faults+=1
            if ffc!=res:
                detected+=1
    
    print("All tested faults: ",all_faults)
    print("Detected faults: ", detected)
    print("Percentage: ",detected/all_faults)
    print(f"Time taken: {(time.perf_counter()-t1):09.3f}")

    t2=time.perf_counter()
    detected=0
    all_faults=0
    for f in d:
        if f.split("@")[1]=='0':
            aux2=x0
        else:
            aux2=x1
        aux1={f.split("@")[0]:aux2}
        res=circuit(a,z,b,c,aux1)
        all_faults+=1
        if ffc!=res:
            detected+=1
            # print("Fault detected: ",e)
    log("Finished comparing")
    print("All tested faults: ",all_faults)
    print("Detected faults: ", detected)
    print("Percentage: ",detected/all_faults)
    print(f"Time taken: {(time.perf_counter()-t2):09.3f}")
    print("#"*100)
    print()

Finished reading bench file:  ./data.nogit/c2670.bench
# 00091.890 - Finished reading bench file
****************************************************************************************************
# of POs:  64
# of fanout stems:  454
# of gate inputs:  1286
All possible faults:  4796
Calculated collapsed faults:  2595
Time taken: 00000.142
****************************************************************************************************
Time taken:  0:00:00.002769
# of PIS:  157
# of POS:  64

All tested faults:  4668
Detected faults:  3631
Percentage:  0.7778491859468724
Time taken: 00021.569
# 00125.276 - Finished comparing
All tested faults:  2595
Detected faults:  2069
Percentage:  0.7973025048169556
Time taken: 00011.681
####################################################################################################

Finished reading bench file:  ./data.nogit/c5315.bench
# 00125.320 - Finished reading bench file
*************************************************************

In [ ]:
ini,s0,sa1=test1("./data.nogit/c17.tests")

Time taken:  0:00:00.002346


In [65]:
from functools import reduce
import operator

def AND(*args):
    return reduce(operator.and_, args)

def OR(*args):
    return reduce(operator.or_, args)

def XOR(*args):
    return reduce(operator.xor_, args)

def NOT(a):
    return ~a

def NAND(*args):
    return ~AND(*args)

def NOR(*args):
    return ~OR(*args)

def XNOR(*args):
    return ~XOR(*args)

def BUFF(a):
    return a

LOGIC_GATES = {
    "AND": AND,
    "NAND": NAND,
    "OR": OR,
    "NOR": NOR,
    "XOR": XOR,
    "NOT": NOT,
    "XNOR": XNOR,
    "BUFF": BUFF,
}
def evaluate_circuit(signals, operations,POs,SA={}):
    """
    Evaluates an acyclic logic circuit.

    signals: dict[int, int]
    operations: dict[int, (str, list[int])]

    Returns updated signals dict.
    """
    for out_sig, (gate, inputs) in operations.items():
        if out_sig in SA:
            signals[out_sig]=SA[out_sig]
            continue
    
        func = LOGIC_GATES[gate]

        input_values = [signals[i] for i in inputs]
        signals[out_sig] = func(*input_values)

    return [signals[x] for x in POs]


In [60]:
a,b,c,d=new_read_bench("./data.nogit/c17.bench")
ini,s0,s1=test1("./data.nogit/c17.tests")
operations={

}
signals={y:ini[x] for x,y in enumerate(b)}
for z in a:
    out,gt,ins=parts(z)
    operations[out]=(gt, ins)

tests=[]
for item in d:
    name, val = item.split("@")
    if val=="0":
        tests.append((name, s0))
    else:
        tests.append((name, s1))

# 00108.382 - Finished reading bench file
['10 = NAND(1, 3)', '11 = NAND(3, 6)', '16 = NAND(2, 11)', '19 = NAND(11, 7)', '22 = NAND(10, 16)', '23 = NAND(16, 19)']
****************************************************************************************************
# of POs:  2
# of fanout stems:  3
# of gate inputs:  9
All possible faults:  34
Calculated collapsed faults:  22
Time taken: 00000.005
****************************************************************************************************
Time taken:  0:00:00.000177


In [66]:
base=forced_signals = evaluate_circuit(
            signals.copy(),
            operations,
            c)

print(base)

[927194937356220562902518033407, 229575789345848977198178363855]


In [ ]:
def run_forced_tests(
    signals_init,
    operations,
    baseline_signals,
    force_tests,
    output_ids,
    num_tests
):
    results = {}

    for sig_id, forced_value in force_tests:
        # Build force mask
        force_masks = {
            sig_id: ((1 << num_tests) - 1) if forced_value else 0
        }

        # Run circuit with force
        forced_signals = evaluate_circuit(
            signals_init.copy(),
            operations,
            force_masks
        )

        # Compare outputs
        diffs = {}
        for out_id in output_ids:
            diff_mask = baseline_signals[out_id] ^ forced_signals[out_id]
            if diff_mask:
                diffs[out_id] = diff_mask

        results[(sig_id, forced_value)] = diffs

    return results


In [24]:
import re
from collections import defaultdict, deque

# Supported gates
LOGIC_GATES = {"AND", "OR", "NOT", "NAND", "XOR", "XNOR", "BUFF"}

def parse_gate_line(line):
    """
    Parses a line like '10 = NAND(1,3)' 
    Returns: output_signal, gate_name, list_of_inputs
    """
    line = line.strip()
    if not line or line.startswith("#") or "INPUT" in line or "OUTPUT" in line:
        return None
    match = re.match(r"(\S+)\s*=\s*(\w+)\((.*)\)", line)
    if not match:
        raise ValueError(f"Invalid gate line: {line}")
    output, gate, inputs = match.groups()
    input_list = [x.strip() for x in inputs.split(",")] if inputs else []
    return output, gate.upper(), input_list


def read_circuit_file(filename):
    """Reads CSV/logic file and returns list of gates and dependencies"""
    gates = []
    signal_sources = {}  # maps signal -> gate that produces it

    with open(filename) as f:
        for line in f:
            parsed = parse_gate_line(line)
            if parsed:
                output, gate, inputs = parsed
                gates.append((output, gate, inputs))
                signal_sources[output] = output  # map output to itself (source)

    return gates, signal_sources


def detect_primary_inputs_outputs(gates):
    """
    Primary inputs: signals used as input but not produced by any gate
    Primary outputs: signals produced but never used as input
    """
    produced = set(out for out, _, _ in gates)
    used = set(inp for _, _, inputs in gates for inp in inputs)

    # test_signals contains all signals provided by the test CSV
    primary_inputs = used - produced
    # primary_inputs &= set(test_signals)  # ensure only test CSV inputs

    primary_outputs = produced - used

    return list(primary_inputs), list(primary_outputs)


def topological_sort(gates):
    """
    Returns gates in topological order (inputs -> outputs)
    Implements Kahn's algorithm
    """
    graph = defaultdict(list)
    indegree = defaultdict(int)
    outputs = {}

    for out, gate, inputs in gates:
        outputs[out] = (gate, inputs)
        for inp in inputs:
            graph[inp].append(out)
            indegree[out] += 1
        if out not in indegree:
            indegree[out] = indegree.get(out, 0)

    # Kahn's algorithm
    queue = deque([n for n in indegree if indegree[n] == 0])
    sorted_nodes = []
    print(queue)

    while queue:
        node = queue.popleft()
        if node in outputs:
            sorted_nodes.append((node, outputs[node][0], outputs[node][1]))
        for neigh in graph[node]:
            indegree[neigh] -= 1
            if indegree[neigh] == 0:
                queue.append(neigh)

    if len(sorted_nodes) != len(gates):
        raise ValueError("Circuit contains a cycle or missing nodes")

    return sorted_nodes


def make_buffers_explicit(sorted_gates):
    """
    Replaces implicit buffer signals by explicit new signals like 'signal.b0'
    Returns updated sorted_gates
    """
    # Count of buffers per original signal
    buffer_count = defaultdict(int)
    updated_gates = []

    for out, gate, inputs in sorted_gates:
        new_inputs = []
        for inp in inputs:
            # If input == output of another gate (or itself), and gate is not a BUFF
            # we can decide if a buffer is needed: let's make every repeated use a buffer
            if inp in [g[0] for g in sorted_gates]:  
                # generate new buffer signal
                bnum = buffer_count[inp]
                buffer_count[inp] += 1
                buf_name = f"{inp}.b{bnum}"
                # insert a buffer gate before using it
                updated_gates.append((buf_name, "BUFF", [inp]))
                new_inputs.append(buf_name)
            else:
                new_inputs.append(inp)
        updated_gates.append((out, gate, new_inputs))

    return updated_gates

In [25]:
# Example usage
# if __name__ == "__main__":
# test_signals = ["a","b","c","d","e"]  # signals from test CSV
gates, signal_sources = read_circuit_file("./data.nogit/c17.bench")
primary_inputs, primary_outputs = detect_primary_inputs_outputs(gates)
# print("Primary inputs:", primary_inputs)
# print("Primary outputs:", primary_outputs)

sorted_gates = topological_sort(gates)
print("Topological sort complete.")

# explicit_gates = make_buffers_explicit(sorted_gates)
# print("Explicit buffers added. Total gates:", len(explicit_gates))


deque([])


ValueError: Circuit contains a cycle or missing nodes